# Ejercicio final de la semana 1

Para demostrar que estás familiarizado con la API de OpenAI y también con Ollama, crea una herramienta que responda a una pregunta técnica
y la explique. ¡Esta es una herramienta que podrás usar durante el curso!

In [1]:
# imports

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI


In [3]:
# constantes

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'gemma4:31b'


In [4]:
# Configura el entorno

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")

openai = OpenAI()

La clave de API parece buena


In [5]:
# Aquí está la pregunta; escribe sobre esto para preguntar algo nuevo

system_prompt = "Necesito que actues como un programador experto en Python. \
Responde lo más detallado posible, dando ejemplos. \
Responde en formato markdown.\n"

user_prompt = """
Explica qué hace este código y por qué:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [6]:
# Haz que gpt-4o-mini responda, con streaming

def python_expert():
    # Llamada al modelo con streaming habilitado
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True  # Habilitar streaming
    )

    # Inicializar variables para mostrar el contenido progresivamente
    response = ""
    display_handle = display(Markdown(""), display_id=True)

    # Procesar cada fragmento de la respuesta
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

    return response


# Llamada a la función
final_response = python_expert()
print("Respuesta final completa:")
print(final_response)


El código que has proporcionado es una expresión que utiliza la declaración `yield from`. Vamos a desglosar cada parte para entenderlo mejor.

## Desglose del Código

### 1. Comprensión de Conjuntos

La parte `{book.get("author") for book in books if book.get("author")}` es una comprensión de conjuntos en Python. Este tipo de comprensiones permite crear un conjunto a partir de una iterable, en este caso, la lista (o colección) `books`.

- `for book in books`: Iteramos sobre cada elemento de `books`. Se asume que `books` es una lista de diccionarios, donde cada diccionario representa un libro.
- `book.get("author")`: Recogemos el valor asociado a la clave `"author"` de cada libro. `get` es un método de diccionario que devuelve el valor para la clave proporcionada si existe; de lo contrario, devuelve `None`.
- `if book.get("author")`: Esta es una cláusula condicional que filtra los resultados. Solo añadimos al conjunto los autores que no son `None`, es decir, aquellos libros que tienen asignado un autor.

Al final de esta parte del código se construye un conjunto que contendrá todos los autores de los libros en `books`, sin elementos duplicados (pues los conjuntos no permiten duplicados).

### 2. `yield from`

La expresión `yield from` se utiliza en generadores para delegar la producción de valores a otro iterable. En este caso, `yield from` se usa con el conjunto que hemos creado anteriormente.

- `yield from <iterable>`: Genera todos los elementos del iterable hasta que se agoten. En este caso, está generando todos los autores recogidos en el conjunto resultante.

### ¿Qué hace el código en conjunto?

1. Itera sobre `books`, que se asume es una lista de diccionarios, extrayendo los autores de cada libro.
2. Filtra aquellos libros que no tienen un autor (es decir, cuyo valor en la clave `"author"` es `None`).
3. Crea un conjunto de todos los autores únicos.
4. Utiliza `yield from` para generar cada uno de estos autores, uno por uno, en lugar de devolver el conjunto completo.

## Ejemplo

Supongamos que tenemos la siguiente lista de libros:

python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": None},
    {"title": "Book 4", "author": "Author A"},  # Duplicado
    {"title": "Book 5", "author": "Author C"}
]


Si aplicamos el código en un contexto como este:

python
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

for author in get_authors(books):
    print(author)


### Salida Esperada

Lo que imprimiría este bucle sería:


Author A
Author B
Author C


Dado que `Author A` se repite en el conjunto, solo aparece una vez en la salida final. 

## Conclusión

El fragmento de código presentado es una forma compacta y eficiente de extraer y generar una lista única de autores de una colección de libros, ignorando aquellos libros que no tienen un autor definido.

Respuesta final completa:
El código que has proporcionado es una expresión que utiliza la declaración `yield from`. Vamos a desglosar cada parte para entenderlo mejor.

## Desglose del Código

### 1. Comprensión de Conjuntos

La parte `{book.get("author") for book in books if book.get("author")}` es una comprensión de conjuntos en Python. Este tipo de comprensiones permite crear un conjunto a partir de una iterable, en este caso, la lista (o colección) `books`.

- `for book in books`: Iteramos sobre cada elemento de `books`. Se asume que `books` es una lista de diccionarios, donde cada diccionario representa un libro.
- `book.get("author")`: Recogemos el valor asociado a la clave `"author"` de cada libro. `get` es un método de diccionario que devuelve el valor para la clave proporcionada si existe; de lo contrario, devuelve `None`.
- `if book.get("author")`: Esta es una cláusula condicional que filtra los resultados. Solo añadimos al conjunto los autores que no son `None`, es decir, aq

In [7]:
# Haz que Llama 3.2 responda

messages = [
    {
        "role": "user",
        "content": (
            "Necesito que actúes como un programador experto en Python. "
            "Responde lo más detallado posible, dando ejemplos. "
            "Responde en formato markdown. "
            "Explica qué hace este código y por qué: "
            "yield from {book.get('author') for book in books if book.get('author')}"
        )
    }
]

payload = {
    "model": MODEL_LLAMA,
    "messages": messages,
    "stream": False
}

response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)

# Procesar la respuesta JSON
response_json = response.json()

# Extraer el contenido de la respuesta
message_content = response_json.get('message', {}).get('content', 'No content received')

# Mostrar en formato Markdown
display(Markdown(message_content))

Hola. Como programador experto en Python, es un placer desglosar esta línea de código para ti. Estamos ante una expresión muy compacta y elegante que combina tres conceptos avanzados de Python: **Set Comprehensions**, **Generadores (yield from)** y el método **`.get()`** de los diccionarios.

A continuación, presento la explicación detallada.

---

# Análisis Técnico: `yield from {book.get('author') for book in books if book.get('author')}`

Esta línea de código se encuentra normalmente dentro de una función generadora. Su objetivo principal es: **extraer una lista de autores únicos de una colección de libros, ignorando aquellos que no tengan autor definido, y devolverlos uno a uno.**

Vamos a diseccionar la sentencia de adentro hacia afuera.

## 1. El Set Comprehension (Comprensión de Conjuntos)
La parte encerrada entre llaves `{ ... }` es una **Set Comprehension**.

```python
{book.get('author') for book in books if book.get('author')}
```

### ¿Qué hace exactamente?
1.  **`for book in books`**: Itera sobre una lista (o iterable) llamada `books`. Se asume que cada `book` es un diccionario.
2.  **`if book.get('author')`**: Esta es la cláusula de filtrado. El método `.get('author')` intenta obtener el valor de la clave `'author'`. 
    *   Si la clave no existe, devuelve `None`.
    *   Si la clave existe pero está vacía (ej. `""`), se evalúa como `False`.
    *   **Resultado:** Solo los libros que tengan un autor válido pasan el filtro.
3.  **`book.get('author')` (al principio)**: Es el valor que se inserta en el conjunto.
4.  **El uso de `{}` (Set)**: A diferencia de las listas `[]`, los conjuntos **no permiten duplicados**. Si tienes 10 libros de "Stephen King", el conjunto final solo contendrá el nombre "Stephen King" una sola vez.

## 2. El operador `yield from`
Esta es una característica introducida en Python 3.3 para simplificar la delegación de generadores.

### ¿Qué hace exactamente?
Normalmente, para devolver elementos de una lista en un generador, harías esto:
```python
for author in authors_set:
    yield author
```

`yield from` es el "atajo" profesional. Le dice a Python: *"Toma este iterable (en este caso, nuestro conjunto de autores) y devuelve cada uno de sus elementos uno por uno hasta que se agoten"*.

**Ventaja:** Es más eficiente y hace que el código sea más legible y conciso.

---

# Ejemplo Completo y Ejecutable

Para que veas esto en acción, he preparado el siguiente script. Copia y ejecútalo en tu entorno de Python:

```python
def get_unique_authors(books):
    """
    Generador que devuelve autores únicos de una lista de libros.
    """
    # Aquí ocurre la magia:
    # 1. Crea un SET con autores únicos y no nulos.
    # 2. Delega la iteración del SET al generador mediante 'yield from'.
    yield from {book.get('author') for book in books if book.get('author')}

# --- Datos de prueba ---
library = [
    {"title": "El Resplandor", "author": "Stephen King"},
    {"title": "It", "author": "Stephen King"},       # Duplicado: No debería aparecer dos veces
    {"title": "1984", "author": "George Orwell"},
    {"title": "El Hobbit", "author": "J.R.R. Tolkien"},
    {"title": "Libro Anónimo", "author": None},       # Filtro: No debería aparecer
    {"title": "Libro Misterioso"},                    # Filtro: No tiene la clave 'author', no debe aparecer
]

print("Lista de autores únicos:")
for author in get_unique_authors(library):
    print(f"- {author}")
```

### Salida esperada:
```text
Lista de autores únicos:
- Stephen King
- George Orwell
- J.R.R. Tolkien
```
*(Nota: El orden puede variar ya que los conjuntos no mantienen el orden de inserción).*

---

# Resumen de "Por qué" se hace así (Justificación de Experto)

Si yo estuviera revisando este código en un *Code Review*, aprobaría esta línea por las siguientes razones:

1.  **Robustez (`.get()`):** Usar `book['author']` lanzaría un error `KeyError` si la clave no existe. `.get()` es la forma segura de acceder a diccionarios cuando no estamos seguros de si la clave está presente.
2.  **Eficiencia de Memoria (`yield from`):** Al usar un generador en lugar de retornar una lista `return [...]`, el programa no carga todos los resultados en la memoria RAM a la vez, sino que los entrega bajo demanda. Esto es vital si la lista de libros fuera de millones de registros.
3.  **Eliminación de Duplicados (`Set`):** La operación de eliminar duplicados en un conjunto es extremadamente rápida ($\mathcal{O}(1)$ en promedio para inserciones y búsquedas), siendo mucho más eficiente que hacer un loop y comprobar `if author not in unique_authors`.
4.  **Concisión:** Logra en una sola línea lo que normalmente requeriría unas 5 o 6 líneas de código tradicional.

### Complejidad Computacional:
*   **Tiempo:** $\mathcal{O}(n)$, donde $n$ es el número de libros. Debemos recorrer la lista una vez.
*   **Espacio:** $\mathcal{O}(u)$, donde $u$ es el número de autores únicos almacenados en el set antes de ser emitidos.